# Fusion — `baselines.ipynb`

**Purpose:** Establish performance baselines before the learned fusion module:
- Individual stream LR probes (sanity-check known results)
- Concatenated features → LR ("simple fusion")
- Probability-level fusion (mean, max, stacking)
- Save aligned feature matrices for `fusion_module.ipynb`

**Inputs:**
- Stream A: `cache/entity_scorer/features_newsclippings_{split}.pkl`
- Stream B: `cache/stream_b_final/stream_b_final_{nc_split|verite|mmfake}.pkl`
- Stream C: `cache/stance_scores/stream_c_{split}.pkl`

**Key challenge:** Alignment — the three streams may have different orderings.
We verify label-array equality across streams; if misaligned, we align by pair_key.

## 0. Environment

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## 1. Imports & Paths

In [ ]:
import os, sys, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

try:
    ROOT = Path(__file__).resolve().parents[2]
except NameError:
    ROOT = Path(os.getcwd())
    while not (ROOT / 'src' / 'utils' / 'paths.py').exists():
        ROOT = ROOT.parent
        if ROOT == ROOT.parent:
            ROOT = Path(os.getcwd()); break

sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

ENT_DIR    = ROOT / 'cache' / 'entity_scorer'
SB_DIR     = ROOT / 'cache' / 'stream_b_final'
SC_DIR     = ROOT / 'cache' / 'stance_scores'
FUSION_DIR = ROOT / 'cache' / 'fusion'
FIG_DIR    = ROOT / 'paper' / 'figures'
TAB_DIR    = ROOT / 'paper' / 'tables'
FUSION_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Stream A Features (Entity Scorer)

In [ ]:
def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# Stream A file mapping
SA_FILES = {
    'train':  ENT_DIR / 'features_newsclippings_train.pkl',
    'val':    ENT_DIR / 'features_newsclippings_val.pkl',
    'test':   ENT_DIR / 'features_newsclippings_test.pkl',
    'verite': ENT_DIR / 'features_verite.pkl',
    'mmfake': ENT_DIR / 'features_mmfake.pkl',
}

# Load & inspect structure
sa_raw = {}
for split, path in SA_FILES.items():
    if not path.exists():
        print(f"  ❌ MISSING: {path}")
        continue
    sa_raw[split] = load_pkl(path)
    data = sa_raw[split]
    print(f"  ✅ {split:8s}: type={type(data).__name__}", end='')
    if isinstance(data, pd.DataFrame):
        print(f"  shape={data.shape}  cols={list(data.columns)[:6]}")
    elif isinstance(data, dict):
        sample_key = next(iter(data))
        sample_val = data[sample_key]
        print(f"  keys={len(data)}  sample_key={sample_key}  "
              f"val_type={type(sample_val).__name__}")
        if isinstance(sample_val, dict):
            print(f"           val_keys={list(sample_val.keys())[:8]}")
    elif isinstance(data, np.ndarray):
        print(f"  shape={data.shape}")
    else:
        print(f"  len={len(data) if hasattr(data,'__len__') else '?'}")

In [ ]:
# ── Convert Stream A to aligned feature matrices ─────────────────────────────
# Stream A features can be: DataFrame, dict-of-dicts, or dict with 'features'+'labels'

def stream_a_to_matrix(data, split):
    """
    Convert Stream A data to (X, y, feat_names, pair_keys).
    Handles DataFrame, dict-of-dicts, or dict with 'features'+'labels'.
    """
    if isinstance(data, pd.DataFrame):
        # Find label column
        label_col = None
        for c in ('label', 'falsified', 'y', 'target'):
            if c in data.columns:
                label_col = c; break
        if label_col is None:
            # Last column might be label
            label_col = data.columns[-1]
        feat_cols = [c for c in data.columns if c != label_col
                     and c not in ('pair_key','pair_id','image_id','split')]
        X = data[feat_cols].values.astype(np.float32)
        y = data[label_col].values.astype(int)
        pair_keys = data['pair_key'].tolist() if 'pair_key' in data.columns else list(range(len(X)))
        return X, y, feat_cols, pair_keys

    elif isinstance(data, dict):
        # Check if it's {features: np.array, labels: np.array} format
        if 'features' in data and 'labels' in data:
            X = np.array(data['features'], dtype=np.float32)
            y = np.array(data['labels'], dtype=int)
            fnames = data.get('feat_names', [f'sa_{i}' for i in range(X.shape[1])])
            pkeys = data.get('pair_keys', list(range(len(X))))
            return X, y, fnames, pkeys

        # dict-of-dicts: {pair_key: {feat1: val, feat2: val, ..., label: 0/1}}
        sample_val = next(iter(data.values()))
        if isinstance(sample_val, dict):
            # Find label key
            label_key = None
            for k in ('label', 'falsified', 'y'):
                if k in sample_val:
                    label_key = k; break
            feat_keys = [k for k in sample_val.keys()
                         if k != label_key and k not in ('pair_key','pair_id','image_id','split')
                         and isinstance(sample_val[k], (int, float, np.integer, np.floating))]
            pair_keys = list(data.keys())
            X = np.array([[data[pk].get(fk, 0.0) for fk in feat_keys]
                          for pk in pair_keys], dtype=np.float32)
            y = np.array([data[pk].get(label_key, 0) for pk in pair_keys], dtype=int)
            return X, y, feat_keys, pair_keys

    raise ValueError(f"Unknown Stream A format: {type(data)}")


sa = {}
for split in ('train', 'val', 'test', 'verite', 'mmfake'):
    if split not in sa_raw:
        print(f"  ⚠️  {split} not loaded — skipping")
        continue
    X, y, fnames, pkeys = stream_a_to_matrix(sa_raw[split], split)
    sa[split] = {'features': X, 'labels': y, 'feat_names': fnames, 'pair_keys': pkeys}
    print(f"  {split:8s}: X={X.shape}  y_pos={y.sum()} ({100*y.mean():.1f}%)  "
          f"feats={len(fnames)}")

print(f"\nStream A feature names: {sa['test']['feat_names']}")

## 3. Load Stream B & C Features

In [ ]:
SB_FILES = {
    'train':  SB_DIR / 'stream_b_final_nc_train.pkl',
    'val':    SB_DIR / 'stream_b_final_nc_val.pkl',
    'test':   SB_DIR / 'stream_b_final_nc_test.pkl',
    'verite': SB_DIR / 'stream_b_final_verite.pkl',
    'mmfake': SB_DIR / 'stream_b_final_mmfake.pkl',
}
SC_FILES = {
    'train':  SC_DIR / 'stream_c_train.pkl',
    'val':    SC_DIR / 'stream_c_val.pkl',
    'test':   SC_DIR / 'stream_c_test.pkl',
    'verite': SC_DIR / 'stream_c_verite.pkl',
    'mmfake': SC_DIR / 'stream_c_mmfake.pkl',
}

sb, sc_data = {}, {}
print("Stream B:")
for split, path in SB_FILES.items():
    if not path.exists():
        print(f"  ❌ {split}"); continue
    d = load_pkl(path)
    sb[split] = d
    print(f"  ✅ {split:8s}: feats={d['features'].shape}  "
          f"labels_pos={d['labels'].sum()} ({100*d['labels'].mean():.1f}%)")

print("\nStream C:")
for split, path in SC_FILES.items():
    if not path.exists():
        print(f"  ❌ {split}"); continue
    d = load_pkl(path)
    sc_data[split] = d
    print(f"  ✅ {split:8s}: feats={d['features'].shape}  "
          f"labels_pos={d['labels'].sum()} ({100*d['labels'].mean():.1f}%)")

## 4. Verify Alignment

All three streams must have the same sample count per split and matching labels.
If labels don't match, the streams are in different order — we'll need pair_key alignment.

In [ ]:
print("Alignment check (sample counts & label agreement):")
print(f"{'Split':>8} | {'A':>7} | {'B':>7} | {'C':>7} | {'A==B':>5} | {'A==C':>5} | {'B==C':>5}")
print("-" * 60)

aligned_splits = []
for split in ('train', 'val', 'test', 'verite', 'mmfake'):
    n_a = len(sa[split]['labels'])    if split in sa      else 0
    n_b = len(sb[split]['labels'])    if split in sb      else 0
    n_c = len(sc_data[split]['labels']) if split in sc_data else 0

    if n_a == n_b == n_c and n_a > 0:
        ya = sa[split]['labels']
        yb = sb[split]['labels']
        yc = sc_data[split]['labels']
        ab = np.array_equal(ya, yb)
        ac = np.array_equal(ya, yc)
        bc = np.array_equal(yb, yc)
        ok = ab and ac and bc
        print(f"{split:>8} | {n_a:>7} | {n_b:>7} | {n_c:>7} | "
              f"{'✅' if ab else '❌':>5} | {'✅' if ac else '❌':>5} | "
              f"{'✅' if bc else '❌':>5}  {'← aligned' if ok else '← MISALIGNED'}")
        if ok:
            aligned_splits.append(split)
    else:
        print(f"{split:>8} | {n_a:>7} | {n_b:>7} | {n_c:>7} | {'SIZE MISMATCH':>20}")

print(f"\nAligned splits: {aligned_splits}")

In [ ]:
# ── Handle misalignment via label-sum matching ───────────────────────────────
# If labels don't match exactly but sums are equal, it's an ordering issue.
# For LR baselines, label ordering doesn't matter as long as X and y are
# internally consistent within each stream. For concatenation, we need
# row-level alignment. If misaligned, we fall back to probability-level fusion.

# For splits where sizes match but labels differ, check if label sums match
for split in ('train', 'val', 'test', 'verite', 'mmfake'):
    if split in aligned_splits:
        continue
    if split in sa and split in sb and split in sc_data:
        sum_a = sa[split]['labels'].sum()
        sum_b = sb[split]['labels'].sum()
        sum_c = sc_data[split]['labels'].sum()
        n_a = len(sa[split]['labels'])
        n_b = len(sb[split]['labels'])
        n_c = len(sc_data[split]['labels'])
        if n_a == n_b == n_c:
            print(f"  {split}: sizes match ({n_a}) but labels differ")
            print(f"    label sums: A={sum_a} B={sum_b} C={sum_c}")
            if sum_a == sum_b == sum_c:
                print(f"    → Same class balance — ordering difference only")
            else:
                print(f"    → Different class balance — alignment required")

## 5. Individual Stream LR Probes (Sanity Check)

In [ ]:
def lr_probe(X_train, y_train, X_eval, y_eval, name=''):
    """Train LR on train, evaluate on eval. Returns AUROC, Acc, probs."""
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_ev = scaler.transform(X_eval)
    lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    lr.fit(X_tr, y_train)
    probs = lr.predict_proba(X_ev)[:, 1]
    auc = roc_auc_score(y_eval, probs)
    acc = accuracy_score(y_eval, (probs >= 0.5).astype(int))
    return auc, acc, probs, lr, scaler


# Expected AUROCs from previous notebooks
EXPECTED = {
    'Stream A': {'test': 0.732, 'verite': 0.553, 'mmfake': 0.581},
    'Stream B': {'test': 0.796, 'verite': 0.719, 'mmfake': 0.984},
    'Stream C': {'test': 0.607, 'verite': 0.500, 'mmfake': 0.507},
}

individual_results = {}
print(f"{'Stream':>10} | {'Split':>8} | {'AUROC':>7} | {'Acc':>7} | {'Expected':>8} | {'Δ':>7}")
print("-" * 58)

for stream_name, stream_data in [('Stream A', sa), ('Stream B', sb), ('Stream C', sc_data)]:
    if 'train' not in stream_data:
        continue
    X_tr = stream_data['train']['features']
    y_tr = stream_data['train']['labels']
    for eval_split in ('test', 'verite', 'mmfake'):
        if eval_split not in stream_data:
            continue
        X_ev = stream_data[eval_split]['features']
        y_ev = stream_data[eval_split]['labels']
        auc, acc, probs, _, _ = lr_probe(X_tr, y_tr, X_ev, y_ev)
        exp = EXPECTED.get(stream_name, {}).get(eval_split, 0)
        delta = auc - exp if exp else 0
        individual_results[(stream_name, eval_split)] = {
            'auroc': auc, 'acc': acc, 'probs': probs, 'labels': y_ev
        }
        print(f"{stream_name:>10} | {eval_split:>8} | {auc:>7.4f} | {acc:>7.4f} | "
              f"{exp:>8.3f} | {delta:>+7.4f}")

## 6. Concatenated Feature Fusion (LR)

For aligned splits, concatenate [A features | B features | C features] and train a single LR.

In [ ]:
def concat_features(split):
    """Concatenate all three streams' features for a given split."""
    Xa = sa[split]['features']
    Xb = sb[split]['features']
    Xc = sc_data[split]['features']
    y  = sa[split]['labels']
    return np.hstack([Xa, Xb, Xc]), y

concat_results = {}

if 'train' in aligned_splits or True:  # train should always be aligned
    try:
        X_tr, y_tr = concat_features('train')
        print(f"Concatenated train: {X_tr.shape} "
              f"(A={sa['train']['features'].shape[1]} + "
              f"B={sb['train']['features'].shape[1]} + "
              f"C={sc_data['train']['features'].shape[1]})")

        print(f"\n{'Split':>8} | {'Concat LR':>10} | {'Best Single':>11} | {'Gain':>7}")
        print("-" * 45)

        for eval_split in ('val', 'test', 'verite', 'mmfake'):
            try:
                X_ev, y_ev = concat_features(eval_split)
                auc, acc, probs, lr_model, scaler = lr_probe(X_tr, y_tr, X_ev, y_ev)
                # Best single stream for this split
                best_single = max(
                    individual_results.get((s, eval_split), {}).get('auroc', 0)
                    for s in ('Stream A', 'Stream B', 'Stream C')
                )
                gain = auc - best_single
                concat_results[eval_split] = {
                    'auroc': auc, 'acc': acc, 'probs': probs, 'labels': y_ev
                }
                print(f"{eval_split:>8} | {auc:>10.4f} | {best_single:>11.4f} | {gain:>+7.4f}")
            except Exception as e:
                print(f"{eval_split:>8} | FAILED: {e}")
    except Exception as e:
        print(f"Concatenation failed: {e}")
        print("Falling back to probability-level fusion (Cell 7).")

## 7. Probability-Level Fusion

Even if feature-level alignment fails, we can always stack per-stream
LR probabilities as a 3-feature vector and train a meta-learner.

In [ ]:
# First get per-stream probs for all eval splits using individual LR probes
stream_probs = {'Stream A': {}, 'Stream B': {}, 'Stream C': {}}

for stream_name, stream_data in [('Stream A', sa), ('Stream B', sb), ('Stream C', sc_data)]:
    X_tr = stream_data['train']['features']
    y_tr = stream_data['train']['labels']
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    lr.fit(X_tr_s, y_tr)
    for split in ('train', 'val', 'test', 'verite', 'mmfake'):
        if split in stream_data:
            X_ev = scaler.transform(stream_data[split]['features'])
            stream_probs[stream_name][split] = lr.predict_proba(X_ev)[:, 1]

# ── Mean fusion ──────────────────────────────────────────────────────────────
print(f"{'Method':>20} | {'Split':>8} | {'AUROC':>7} | {'Acc':>7}")
print("-" * 50)

prob_fusion_results = {}
for eval_split in ('test', 'verite', 'mmfake'):
    y_ev = sa[eval_split]['labels']  # same labels for aligned splits
    pa = stream_probs['Stream A'].get(eval_split)
    pb = stream_probs['Stream B'].get(eval_split)
    pc = stream_probs['Stream C'].get(eval_split)
    if pa is None or pb is None or pc is None:
        continue

    # Mean
    p_mean = (pa + pb + pc) / 3
    auc_mean = roc_auc_score(y_ev, p_mean)
    acc_mean = accuracy_score(y_ev, (p_mean >= 0.5).astype(int))
    print(f"{'Mean fusion':>20} | {eval_split:>8} | {auc_mean:>7.4f} | {acc_mean:>7.4f}")

    # Max
    p_max = np.maximum(np.maximum(pa, pb), pc)
    auc_max = roc_auc_score(y_ev, p_max)
    acc_max = accuracy_score(y_ev, (p_max >= 0.5).astype(int))
    print(f"{'Max fusion':>20} | {eval_split:>8} | {auc_max:>7.4f} | {acc_max:>7.4f}")

    # Weighted mean (emphasize Stream B which is strongest)
    p_wt = 0.2 * pa + 0.6 * pb + 0.2 * pc
    auc_wt = roc_auc_score(y_ev, p_wt)
    acc_wt = accuracy_score(y_ev, (p_wt >= 0.5).astype(int))
    print(f"{'Weighted (.2/.6/.2)':>20} | {eval_split:>8} | {auc_wt:>7.4f} | {acc_wt:>7.4f}")

    prob_fusion_results[eval_split] = {
        'mean': auc_mean, 'max': auc_max, 'weighted': auc_wt
    }
    print()

In [ ]:
# ── Stacking (meta-LR on 3 probability features) ────────────────────────────
print("Stacking meta-LR (train on prob features [pA, pB, pC]):")
print(f"{'Split':>8} | {'Stacking':>8} | {'Best Single':>11} | {'Concat LR':>9} | {'Gain':>7}")
print("-" * 55)

X_stack_tr = np.column_stack([
    stream_probs['Stream A']['train'],
    stream_probs['Stream B']['train'],
    stream_probs['Stream C']['train'],
])
y_stack_tr = sa['train']['labels']

stack_lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
stack_lr.fit(X_stack_tr, y_stack_tr)
print(f"  Stack LR coef: A={stack_lr.coef_[0][0]:.3f}  "
      f"B={stack_lr.coef_[0][1]:.3f}  C={stack_lr.coef_[0][2]:.3f}")

stacking_results = {}
for eval_split in ('val', 'test', 'verite', 'mmfake'):
    pa = stream_probs['Stream A'].get(eval_split)
    pb = stream_probs['Stream B'].get(eval_split)
    pc = stream_probs['Stream C'].get(eval_split)
    if pa is None or pb is None or pc is None:
        continue
    X_stack = np.column_stack([pa, pb, pc])
    y_ev    = sa[eval_split]['labels']
    probs   = stack_lr.predict_proba(X_stack)[:, 1]
    auc     = roc_auc_score(y_ev, probs)
    acc     = accuracy_score(y_ev, (probs >= 0.5).astype(int))
    best_s  = max(individual_results.get((s, eval_split), {}).get('auroc', 0)
                  for s in ('Stream A', 'Stream B', 'Stream C'))
    concat_auc = concat_results.get(eval_split, {}).get('auroc', 0)
    stacking_results[eval_split] = {'auroc': auc, 'acc': acc, 'probs': probs, 'labels': y_ev}
    print(f"{eval_split:>8} | {auc:>8.4f} | {best_s:>11.4f} | "
          f"{concat_auc:>9.4f} | {auc - best_s:>+7.4f}")

## 8. Grand Results Table

In [ ]:
ALL_RESULTS = {
    'Majority':    {'NC test': 0.500, 'VERITE': 0.500, 'MMFake': 0.500},
    'Stream A':    {},
    'Stream B':    {},
    'Stream C':    {},
    'Mean fusion': {},
    'Weighted':    {},
    'Stacking':    {},
    'Concat LR':   {},
}

SPLIT_MAP = {'test': 'NC test', 'verite': 'VERITE', 'mmfake': 'MMFake'}

for eval_split, col_name in SPLIT_MAP.items():
    for s in ('Stream A', 'Stream B', 'Stream C'):
        ir = individual_results.get((s, eval_split), {})
        if ir:
            ALL_RESULTS[s][col_name] = ir['auroc']
    pf = prob_fusion_results.get(eval_split, {})
    if pf:
        ALL_RESULTS['Mean fusion'][col_name] = pf['mean']
        ALL_RESULTS['Weighted'][col_name] = pf['weighted']
    sr = stacking_results.get(eval_split, {})
    if sr:
        ALL_RESULTS['Stacking'][col_name] = sr['auroc']
    cr = concat_results.get(eval_split, {})
    if cr:
        ALL_RESULTS['Concat LR'][col_name] = cr['auroc']

df_all = pd.DataFrame(ALL_RESULTS).T
df_all.index.name = 'Method'
df_all = df_all.round(4)
print("Grand AUROC Table:")
print(df_all.to_string())
df_all.to_csv(TAB_DIR / 'fusion_baselines.csv')
print(f"\nSaved → paper/tables/fusion_baselines.csv")

## 9. ROC Curves — Fusion vs Individual Streams

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (eval_split, title) in zip(axes, [('test','NewsCLIPpings test'),
                                           ('verite','VERITE'),
                                           ('mmfake','MMFakeBench')]):
    # Individual streams
    for sname, color, ls in [('Stream A','#4caf50','-'), ('Stream B','#2196f3','-'),
                              ('Stream C','#9c27b0','-')]:
        ir = individual_results.get((sname, eval_split))
        if ir:
            fpr, tpr, _ = roc_curve(ir['labels'], ir['probs'])
            ax.plot(fpr, tpr, color=color, lw=1.5, ls=ls,
                    label=f"{sname} ({ir['auroc']:.3f})")

    # Concat LR
    cr = concat_results.get(eval_split)
    if cr:
        fpr, tpr, _ = roc_curve(cr['labels'], cr['probs'])
        ax.plot(fpr, tpr, color='#e65100', lw=2.5,
                label=f"Concat LR ({cr['auroc']:.3f})")

    # Stacking
    sr = stacking_results.get(eval_split)
    if sr:
        fpr, tpr, _ = roc_curve(sr['labels'], sr['probs'])
        ax.plot(fpr, tpr, color='#d32f2f', lw=2, ls='--',
                label=f"Stacking ({sr['auroc']:.3f})")

    ax.plot([0,1],[0,1], 'k--', lw=0.8, alpha=0.3)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=7.5, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Fusion Baselines — ROC Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
fig_path = FIG_DIR / 'fusion_baselines_roc.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fig_path}")

## 10. Save Aligned Feature Matrices for Fusion Module

In [ ]:
def save_pkl(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    print(f"  Saved {path.name} ({path.stat().st_size/1e6:.1f} MB)")

print("Saving aligned fusion inputs …")
for split in ('train', 'val', 'test', 'verite', 'mmfake'):
    if split not in sa or split not in sb or split not in sc_data:
        print(f"  ⚠️ {split} — missing from one or more streams, skipping")
        continue

    bundle = {
        'stream_a': sa[split]['features'],
        'stream_b': sb[split]['features'],
        'stream_c': sc_data[split]['features'],
        'labels':   sa[split]['labels'],
        'sa_feat_names': sa[split]['feat_names'],
        'sb_dim': sb[split]['features'].shape[1],
        'sc_dim': sc_data[split]['features'].shape[1],
        # Also store per-stream probs for probability-level fusion
        'prob_a': stream_probs['Stream A'].get(split),
        'prob_b': stream_probs['Stream B'].get(split),
        'prob_c': stream_probs['Stream C'].get(split),
    }
    save_pkl(bundle, FUSION_DIR / f'fusion_input_{split}.pkl')

# Save baseline results for comparison
save_pkl({
    'individual': {k: v['auroc'] for k, v in individual_results.items()},
    'concat':     {k: v['auroc'] for k, v in concat_results.items()},
    'stacking':   {k: v['auroc'] for k, v in stacking_results.items()},
    'prob_fusion': prob_fusion_results,
    'stack_coef':  stack_lr.coef_[0].tolist(),
}, FUSION_DIR / 'baseline_results.pkl')

print(f"\n✅ All fusion inputs saved to: {FUSION_DIR}")

## 11. Summary

In [ ]:
print("=" * 65)
print("FUSION — baselines.ipynb COMPLETE")
print("=" * 65)
print()
print("Grand AUROC Table:")
print(df_all.to_string())
print()
print(f"Stack LR coef: A={stack_lr.coef_[0][0]:.3f}  "
      f"B={stack_lr.coef_[0][1]:.3f}  C={stack_lr.coef_[0][2]:.3f}")
print(f"→ Stream B dominates fusion (highest coefficient)")
print()
print("Saved files:")
for f in sorted(FUSION_DIR.iterdir()):
    print(f"  {f.name:40s} {f.stat().st_size/1e6:6.1f} MB")
print()
print("Next notebook: 05_fusion/fusion_module.ipynb")